In [2]:
import math
from pathlib import Path
import time
# import warnings
# warnings.filterwarnings('ignore')

import os
os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

import torch
import numpy as np
from torch.utils.data import DataLoader

from chop.models.chronos2.pipeline import Chronos2Pipeline
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN          = 1440   # context length per time series
N_FEATURES       = 20     # number of variates per sample
FORECAST_HORIZON = 96     # prediction length
BATCH_SIZES      = [20, 40, 60, 80, 100]  # number of multivariate samples per batch
WARMUP_RUNS      = 3
BENCH_RUNS       = 5
MODEL_ID         = 'amazon/chronos-2'

print(f'Device: {DEVICE}')
print(f'Sequence length: {SEQ_LEN}')
print(f'Number of features: {N_FEATURES}')
print(f'Forecast horizon: {FORECAST_HORIZON}')

Device: cuda
Sequence length: 1440
Number of features: 20
Forecast horizon: 96


In [3]:
from chop.models import get_model

model = get_model("chronos-2", pretrained=True, model_id=MODEL_ID, use_sparse_group_attn=False, _attn_implementation="sdpa")
model = model.to(DEVICE).to(torch.bfloat16)
pipeline = Chronos2Pipeline(model=model)
model.eval()

OUTPUT_PATCH_SIZE = model.chronos_config.output_patch_size
NUM_OUTPUT_PATCHES = math.ceil(FORECAST_HORIZON / OUTPUT_PATCH_SIZE)
FUTURE_LEN = NUM_OUTPUT_PATCHES * OUTPUT_PATCH_SIZE

print(f'✓ Model loaded successfully from {MODEL_ID}')
print(f'  Model type:        {type(model).__name__}')
print(f'  Output patch size: {OUTPUT_PATCH_SIZE}')
print(f'  Num output patches for horizon={FORECAST_HORIZON}: {NUM_OUTPUT_PATCHES}')


✓ Model loaded successfully from amazon/chronos-2
  Model type:        Chronos2Model
  Output patch size: 16
  Num output patches for horizon=96: 6


In [4]:
# Benchmark function 

import pandas as pd


def fev_bench_inference_time(model, model_name, task_configs, batch_sizes, output_dir="artifacts"):
    import fev

    # Define benchmark
    benchmark = fev.Benchmark.from_list(task_configs)

    # Run benchmark for each model and batch size
    summaries = []
    inference_times = {}
    print(f'\n=== Benchmarking model: {model_name} ===')

    # Create pipeline for the model
    pipeline = Chronos2Pipeline(model=model)

    for task in benchmark.tasks:
        print(f'\n--- Task: {task.task_name} ---')

        inference_times[task.task_name] = {}

        for batch_size in batch_sizes:
            predictions, inference_time = pipeline.predict_fev(task, batch_size)
            summary = task.evaluation_summary(predictions, model_name)
            summaries.append(summary)

            inference_times[task.task_name][batch_size] = inference_time
            
            inference_time_per_sample = inference_time / batch_size
            print(f'Batch size: {batch_size} | Inference time per sample: {inference_time_per_sample:.4f} sec')

    # ── Save results to CSV ───────────────────────────────────────────────────
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Summaries CSV
    summaries_df = pd.DataFrame(summaries)
    summaries_path = Path(output_dir) / f"{model_name}_summaries.csv"
    summaries_df.to_csv(summaries_path, index=False)
    print(f'\nSaved summaries → {summaries_path}')

    # Inference times CSV  (rows = tasks, columns = batch sizes)
    inference_rows = []
    for task_name, batch_dict in inference_times.items():
        for batch_size, t in batch_dict.items():
            inference_rows.append({
                "task": task_name,
                "batch_size": batch_size,
                "inference_time_s": t,
                "inference_time_per_sample_s": t / batch_size,
            })
    inference_df = pd.DataFrame(inference_rows)
    inference_path = Path(output_dir) / f"{model_name}_inference_times.csv"
    inference_df.to_csv(inference_path, index=False)
    print(f'Saved inference times → {inference_path}')

    return summaries_df, inference_df


In [9]:
# Benchmark on baseline chronos2 
tasks_configs = [
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "m4_hourly",
        "horizon": 24,
    },
]

BATCH_SIZES = [20, 40, 60, 80, 100]

fev_bench_inference_time(
    model=model,
    model_name="Baseline",
    task_configs=tasks_configs,
    batch_sizes=BATCH_SIZES,
)


=== Benchmarking model: Baseline ===

--- Task: m4_hourly ---
Batch size: 20 | Inference time per sample: 0.0555 sec
Batch size: 40 | Inference time per sample: 0.0172 sec
Batch size: 60 | Inference time per sample: 0.0119 sec
Batch size: 80 | Inference time per sample: 0.0092 sec
Batch size: 100 | Inference time per sample: 0.0082 sec

Saved summaries → artifacts/Baseline_summaries.csv
Saved inference times → artifacts/Baseline_inference_times.csv


(  model_name                dataset_path dataset_config  horizon  num_windows  \
 0   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 1   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 2   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 3   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 4   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 
    initial_cutoff  window_step_size  min_context_length max_context_length  \
 0             -24                24                   1               None   
 1             -24                24                   1               None   
 2             -24                24                   1               None   
 3             -24                24                   1               None   
 4             -24                24                   1               None   
 
    seasonality  ... static_co

In [7]:
bsr_model = get_model("chronos-2", pretrained=True, model_id=MODEL_ID, use_sparse_group_attn=True, _attn_implementation="sdpa")
bsr_model.use_sparse_group_attn = True
bsr_model = bsr_model.to(DEVICE).to(torch.bfloat16)
pipeline = Chronos2Pipeline(model=bsr_model)
bsr_model.eval()

fev_bench_inference_time(
    model=bsr_model,
    model_name="BSR",
    task_configs=tasks_configs,
    batch_sizes=BATCH_SIZES,
)


=== Benchmarking model: BSR ===

--- Task: m4_hourly ---
Batch size: 20 | Inference time per sample: 0.0576 sec
Batch size: 40 | Inference time per sample: 0.0167 sec
Batch size: 60 | Inference time per sample: 0.0109 sec
Batch size: 80 | Inference time per sample: 0.0078 sec
Batch size: 100 | Inference time per sample: 0.0069 sec

Saved summaries → artifacts/BSR_summaries.csv
Saved inference times → artifacts/BSR_inference_times.csv


(  model_name                dataset_path dataset_config  horizon  num_windows  \
 0        BSR  autogluon/chronos_datasets      m4_hourly       24            1   
 1        BSR  autogluon/chronos_datasets      m4_hourly       24            1   
 2        BSR  autogluon/chronos_datasets      m4_hourly       24            1   
 3        BSR  autogluon/chronos_datasets      m4_hourly       24            1   
 4        BSR  autogluon/chronos_datasets      m4_hourly       24            1   
 
    initial_cutoff  window_step_size  min_context_length max_context_length  \
 0             -24                24                   1               None   
 1             -24                24                   1               None   
 2             -24                24                   1               None   
 3             -24                24                   1               None   
 4             -24                24                   1               None   
 
    seasonality  ... static_co

In [11]:
import torch
import time
from chop.models import get_model # Assuming this is where get_model lives in mase
from chop.models.chronos2.pipeline import Chronos2Pipeline
from transformers import AutoConfig

from pathlib import Path

import os
os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

# --- Config ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "amazon/chronos-2"

def setup_models():
    print("Loading Dense (Baseline) Model...")
    # Load exactly as specified in the notebook
    dense_model = get_model("chronos-2", pretrained=True, model_id=MODEL_ID, use_sparse_group_attn=False, _attn_implementation="sdpa")
    dense_model = dense_model.to(DEVICE).to(torch.bfloat16)
    dense_model.eval()
    
    print("Loading Sparse (BSR) Model...")
    # Load exactly as specified in the notebook
    sparse_config = AutoConfig.from_pretrained(MODEL_ID)
    sparse_config.use_sparse_group_attn = True
    sparse_config._attn_implementation = "sdpa"
    sparse_model = get_model("chronos-2", pretrained=True, model_id=MODEL_ID, use_sparse_group_attn=True, _attn_implementation="sdpa")
    sparse_model = sparse_model.to(DEVICE).to(torch.bfloat16)
    sparse_model.eval()
    
    return dense_model, sparse_model

def generate_fake_data(batch_size, context_length=1440, group_size=1):
    """
    Generates synthetic tensors to stress-test the attention layers.
    group_size=1 simulates purely univariate tasks (Best case for Sparse BSR).
    group_size=2 simulates pairs of multivariate series.
    """
    # 1. Fake Context 
    context = torch.randn(batch_size, context_length, dtype=torch.bfloat16, device=DEVICE)
    
    # 2. Fake Mask 
    context_mask = torch.ones(batch_size, context_length, dtype=torch.bfloat16, device=DEVICE)
    
    # 3. Fake Group IDs
    num_groups = max(1, batch_size // group_size)
    group_ids = torch.arange(num_groups, device=DEVICE).repeat_interleave(group_size)
    
    if len(group_ids) < batch_size:
        pad = torch.full((batch_size - len(group_ids),), num_groups, device=DEVICE)
        group_ids = torch.cat([group_ids, pad])
        
    return context, context_mask, group_ids

def run_pure_gpu_benchmark(model, context, context_mask, group_ids, num_runs=10):
    # Warmup
    for _ in range(3):
        with torch.no_grad():
            model(context=context, context_mask=context_mask, group_ids=group_ids)
            
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    
    # Use CUDA events for hyper-accurate GPU timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    start_event.record()
    for _ in range(num_runs):
        with torch.no_grad():
            model(context=context, context_mask=context_mask, group_ids=group_ids)
    end_event.record()
    
    torch.cuda.synchronize()
    
    avg_latency_ms = start_event.elapsed_time(end_event) / num_runs
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    
    return avg_latency_ms, peak_vram_mb

# ==========================================
# Run the Scaling Test
# ==========================================
model_dense, model_sparse = setup_models()

# Push the batch size to highlight the O(B^2) bottleneck
BATCH_SIZES = [100, 300, 500]
CONTEXT_LEN = 1440
GROUP_SIZE = 2 # Simulate pairs of multivariate series

print(f"\n--- Starting Extreme Scaling Benchmark ---")
print(f"Context Length: {CONTEXT_LEN} | Group Size: {GROUP_SIZE}\n")

for b in BATCH_SIZES:
    print(f"Testing Batch Size: {b}...")
    context, context_mask, group_ids = generate_fake_data(b, CONTEXT_LEN, GROUP_SIZE)
    
    try:
        # Test Sparse First
        sparse_lat, sparse_vram = run_pure_gpu_benchmark(model_sparse, context, context_mask, group_ids)
        print(f"  [SPARSE] Latency: {sparse_lat:>6.1f} ms | VRAM: {sparse_vram:>6.1f} MB")
        
        # Test Dense Second (This might OOM at high batch sizes!)
        dense_lat, dense_vram = run_pure_gpu_benchmark(model_dense, context, context_mask, group_ids)
        print(f"  [DENSE]  Latency: {dense_lat:>6.1f} ms | VRAM: {dense_vram:>6.1f} MB")
        
        print(f"  -> Speedup: {dense_lat / sparse_lat:.2f}x | VRAM Saved: {dense_vram - sparse_vram:.1f} MB\n")
        
    except torch.cuda.OutOfMemoryError:
        print(f"  [DENSE]  💥 OUT OF MEMORY (OOM) 💥")
        print(f"  -> Sparse BSR successfully survived a batch size that crashed the baseline!\n")
        torch.cuda.empty_cache() # Clear the VRAM so the next loop can run

Loading Dense (Baseline) Model...
Loading Sparse (BSR) Model...

--- Starting Extreme Scaling Benchmark ---
Context Length: 1440 | Group Size: 2

Testing Batch Size: 100...
  [SPARSE] Latency:  266.7 ms | VRAM: 2509.6 MB
  [DENSE]  Latency:  307.3 ms | VRAM: 2652.9 MB
  -> Speedup: 1.15x | VRAM Saved: 143.3 MB

Testing Batch Size: 300...
  [SPARSE] Latency:  786.4 ms | VRAM: 2857.9 MB
  [DENSE]  Latency: 1219.2 ms | VRAM: 3804.9 MB
  -> Speedup: 1.55x | VRAM Saved: 947.1 MB

Testing Batch Size: 500...
  [SPARSE] Latency: 1317.3 ms | VRAM: 3219.3 MB
  [DENSE]  Latency: 13029.2 ms | VRAM: 5757.8 MB
  -> Speedup: 9.89x | VRAM Saved: 2538.6 MB

